In [ ]:
import numpy as np
import pandas as pd
#import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, RNN
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

In [ ]:
# ??c d? li?u t? file CSV (c? th? thay b?ng file c?a b?n)
data = pd.read_csv('D:\DeTaiNam2024\SoLieuHong\HPG.csv')
# Ki?m tra ti?u ?? c?c c?t
data.head()

In [ ]:
 data = data[['close']]
 data.dropna(inplace=True)

In [ ]:
# Chu?n b? d? li?u
def prepare_data(data, time_steps=60):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data)

    X, y = [], []
    for i in range(time_steps, len(scaled_data)):
        X.append(scaled_data[i-time_steps:i, 0])
        y.append(scaled_data[i, 0])
    X, y = np.array(X), np.array(y)

    # Reshape X ?? ph? h?p v?i RNN [samples, time_steps, features]
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))
    return X, y, scaler


In [ ]:
X, y, scaler = prepare_data(data)

In [ ]:
X

In [ ]:
class sLSTMCell(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super(sLSTMCell, self).__init__(**kwargs)
        self.units = units
        self.state_size = [self.units, self.units]  # [hidden_state, normalizer_state]

    def build(self, input_shape):
        # Kh?i t?o tr?ng s? cho c?c c?ng
        self.W_f = self.add_weight(shape=(input_shape[-1], self.units),
                                   initializer='glorot_uniform',
                                   name='W_f')
        self.U_f = self.add_weight(shape=(self.units, self.units),
                                   initializer='orthogonal',
                                   name='U_f')
        self.b_f = self.add_weight(shape=(self.units,),
                                   initializer='zeros',
                                   name='b_f')

        self.W_i = self.add_weight(shape=(input_shape[-1], self.units),
                                   initializer='glorot_uniform',
                                   name='W_i')
        self.U_i = self.add_weight(shape=(self.units, self.units),
                                   initializer='orthogonal',
                                   name='U_i')
        self.b_i = self.add_weight(shape=(self.units,),
                                   initializer='zeros',
                                   name='b_i')

        self.W_c = self.add_weight(shape=(input_shape[-1], self.units),
                                   initializer='glorot_uniform',
                                   name='W_c')
        self.U_c = self.add_weight(shape=(self.units, self.units),
                                   initializer='orthogonal',
                                   name='U_c')
        self.b_c = self.add_weight(shape=(self.units,),
                                   initializer='zeros',
                                   name='b_c')

        self.W_o = self.add_weight(shape=(input_shape[-1], self.units),
                                   initializer='glorot_uniform',
                                   name='W_o')
        self.U_o = self.add_weight(shape=(self.units, self.units),
                                   initializer='orthogonal',
                                   name='U_o')
        self.b_o = self.add_weight(shape=(self.units,),
                                   initializer='zeros',
                                   name='b_o')
        super(sLSTMCell, self).build(input_shape)

    def call(self, inputs, states):
        h_prev, n_prev = states

        # Forget gate: s? d?ng sigmoid ho?c exp
       # f = tf.keras.activations.sigmoid(tf.matmul(inputs, self.W_f) + tf.matmul(h_prev, self.U_f) + self.b_f)
        f = tf.exp(tf.matmul(inputs, self.W_f) + tf.matmul(h_prev, self.U_f) + self.b_f)

        # Input gate: s? d?ng (sigmoid)^2 ho?c (exp)^2
      #  i = tf.keras.activations.sigmoid(tf.matmul(inputs, self.W_i) + tf.matmul(h_prev, self.U_i) + self.b_i)
       # i = tf.square(i)  # (?(x))^2
        i = tf.exp(tf.matmul(inputs, self.W_i) + tf.matmul(h_prev, self.U_i) + self.b_i)
        # Candidate cell state
        c_tilde = tf.keras.activations.tanh(tf.matmul(inputs, self.W_c) + tf.matmul(h_prev, self.U_c) + self.b_c)

        # Cell state update
        c = f * states[0] + i * c_tilde

        # Normalizer state update
        n = f * n_prev + i

        # Output gate
        o = tf.keras.activations.sigmoid(tf.matmul(inputs, self.W_o) + tf.matmul(h_prev, self.U_o) + self.b_o)

        # Normalized hidden state
        h_tilde = c * tf.math.reciprocal(n + 1e-7)  # th?m epsilon ?? tr?nh chia cho 0
        h = o * h_tilde

        return h, [h, n]

class sLSTM(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super(sLSTM, self).__init__(**kwargs)
        self.units = units
        self.cell = sLSTMCell(units)

    def call(self, inputs):
        return tf.keras.layers.RNN(self.cell, return_sequences=False, return_state=False)(inputs)

    def get_config(self):
        config = super(sLSTM, self).get_config()
        config.update({'units': self.units})
        return config


In [ ]:
# X?y d?ng m? h?nh
def build_model(time_steps, units=50):
    inputs = Input(shape=(time_steps, 1))
    x = sLSTM(units)(inputs)
    x = Dense(1)(x)
    model = Model(inputs, x)
    model.compile(optimizer=Adam(learning_rate=2e-02), loss='mean_squared_error')
    return model
# Kh?i t?o m? h?nh
model = build_model(time_steps=60, units=50)
model.summary()


In [ ]:

# Chia d? li?u th?nh t?p hu?n luy?n v? ki?m tra
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# X?y d?ng m? h?nh
model = build_model(time_steps=60, units=50)

# Hu?n luy?n m? h?nh
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100, batch_size=32,
                    validation_data=(X_test, y_test),
                    callbacks=[early_stop])


In [ ]:

# D? ?o?n tr?n t?p ki?m tra
y_pred = model.predict(X_test)

# ??o chu?n h?a ?? l?y gi? tr? th?c
y_pred_inv = scaler.inverse_transform(y_pred)
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

# T?nh to?n RMSE
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
print(f'Root Mean Squared Error (RMSE): {rmse}')
mse = mean_squared_error(y_test_inv, y_pred_inv)
print(f'Mean Squared Error (MSE): {mse}')
# V? ?? th? so s?nh gi? th?c v? gi? d? ?o?n
plt.figure(figsize=(14, 5))
plt.plot(y_test_inv, color='blue', label='Gi? Th?c')
plt.plot(y_pred_inv, color='red', label='Gi? D? ?o?n')
plt.title('D? ?o?n Gi? C? Phi?u HPG S? D?ng sLSTM')
plt.xlabel('Th?i Gian')
plt.ylabel('Gi? C? Phi?u')
plt.legend()
plt.show()
